In [ ]:
import torch
import torch.nn as nn
import sys
import os

# Check CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
# Define PA-CBAM modules
class AddCoords(nn.Module):
    """Coordinate Injection Layer (CoordConv-style)."""
    def __init__(self, with_r=False):
        super().__init__()
        self.with_r = with_r
    def forward(self, x):
        b, c, h, w = x.shape
        device, dtype = x.device, x.dtype
        y_coords = torch.linspace(-1, 1, h, device=device, dtype=dtype)
        x_coords = torch.linspace(-1, 1, w, device=device, dtype=dtype)
        yy, xx = torch.meshgrid(y_coords, x_coords, indexing='ij')
        xx = xx.unsqueeze(0).unsqueeze(0).expand(b, 1, h, w)
        yy = yy.unsqueeze(0).unsqueeze(0).expand(b, 1, h, w)
        coords = [x, xx, yy]
        if self.with_r:
            coords.append(torch.sqrt(xx ** 2 + yy ** 2))
        return torch.cat(coords, dim=1)

class ChannelAttentionPA(nn.Module):
    """Channel Attention for PA-CBAM (fixed reduction=16)."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False)
        )
    def forward(self, x):
        return x * torch.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttentionPA(nn.Module):
    """Position-Aware Spatial Attention (4-channel input: avg, max, x-coord, y-coord)."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(4, 1, kernel_size, padding=kernel_size // 2, bias=False)
    def forward(self, x, coords):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * torch.sigmoid(self.conv(torch.cat([avg_out, max_out, coords], dim=1)))

class PACBAM(nn.Module):
    """Position-Aware CBAM (PA-CBAM):
    Injects explicit (x, y) coordinate encoding into the spatial attention branch.
    Channel attention uses fixed reduction=16 to match CBAM baseline.
    """
    def __init__(self, kernel_size=7, reduction=16):
        super().__init__()
        self.kernel_size = kernel_size
        self.reduction = reduction
        self.channel_attention = None
        self.spatial_attention = SpatialAttentionPA(kernel_size)
        self.add_coords = AddCoords(with_r=False)
    def forward(self, x):
        if self.channel_attention is None:
            self.channel_attention = ChannelAttentionPA(x.shape[1], self.reduction).to(x.device)
        x = self.channel_attention(x)
        dummy = torch.zeros(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device, dtype=x.dtype)
        coords = self.add_coords(dummy)[:, 1:3, :, :]  # x-coord, y-coord channels
        x = self.spatial_attention(x, coords)
        return x

In [ ]:
# Register PA-CBAM with ultralytics
import ultralytics.nn.tasks as tasks
import ultralytics.nn.modules as modules

for cls in [AddCoords, ChannelAttentionPA, SpatialAttentionPA, PACBAM]:
    setattr(tasks, cls.__name__, cls)
    setattr(modules, cls.__name__, cls)
    for s in ("block","conv","head"):
        if hasattr(modules,s):
            setattr(getattr(modules,s), cls.__name__, cls)
    sys.modules['__main__'].__dict__[cls.__name__] = cls

torch.serialization.add_safe_globals([AddCoords, ChannelAttentionPA, SpatialAttentionPA, PACBAM])
print("PACBAM registered!")

In [ ]:
# UPDATE THIS PATH to your local dataset location
DATASET = "path/to/your/helmet-detection-dataset/archive7_extracted"

# Create data.yaml
content = f"""train: {DATASET}/images/train
val: {DATASET}/images/val
test: {DATASET}/images/test
nc: 2
names: ['helmet', 'non-helmet']
"""
with open("data.yaml", "w") as f:
    f.write(content)

# PA-CBAM YAML architecture
yaml_text = """nc: 2
scales:
  n: [0.33, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, PACBAM, [7]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, PACBAM, [7]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [-1, 1, PACBAM, [7]]
  - [[16, 20, 24], 1, Detect, [nc]]
"""
with open("pacbam.yaml", "w") as f:
    f.write(yaml_text)

print("Both YAML files created!")

In [ ]:
from ultralytics import YOLO

RUN_NAME = "pacbam"
LAST_PT = f"runs/{RUN_NAME}/weights/last.pt"

if os.path.exists(LAST_PT):
    print(f"Resuming from checkpoint: {LAST_PT}")
    model = YOLO(LAST_PT)
    model.train(resume=True, device=0)
else:
    print("Starting fresh training...")
    model = YOLO("pacbam.yaml")
    model.load("yolov8n.pt")
    model.train(
        data="data.yaml",
        epochs=100,
        batch=16,
        imgsz=640,
        optimizer="SGD",
        lr0=0.01,
        cos_lr=True,
        amp=True,
        close_mosaic=10,
        patience=50,
        workers=2,
        save=True,
        save_period=10,
        project="runs",
        name=RUN_NAME,
        exist_ok=True,
        plots=True,
        device=0  # RTX 4060
    )

print("TRAINING DONE!")